In [55]:
import pandas as pd
import json
import importlib

# load the data

In [2]:
concept_root = "../data/concept/"
out_concept_root = "../data/outside_concept/"
response_root = "../data/respondent/"

In [3]:
# take the concepts 
with open(concept_root + 'new_cid_concept_us_food.json', 'r') as f:
    food_concepts = json.load(f)

# all concepts 
all_us_food_concepts = pd.read_excel(out_concept_root + '0407_cleaned_us_food_concepts.xlsx')

# open transformed
with open(response_root + 'transformed_0407_id_normal_interview.json', 'r', encoding='utf-8') as f:
    transformed_respondent = json.load(f)

# the similarity function

give a function/ the code, which is the similarity search 

input:
content to search, list of content to be searched, top_n （n most relevant ones）, bottom_m (m least relevant ones)

output: 
top n most relevant 
bottom m least relevant  


the algo should be optimal (I don't mind do the embedding for all the available input first ), be fast. 
use "sentence-transformers/all-MiniLM-L6-v2"


In [52]:
import sys
sys.path.append("../")
from models import similar as sm

In [56]:
importlib.reload(sm)

<module 'models.similar' from 'c:\\Users\\Yuding.Duan\\OneDrive - Ipsos\\3. self_projects\\llm_synthetic\\combination\\..\\models\\similar.py'>

In [60]:
# Example usage with caching:
corpus = all_us_food_concepts['ConceptText'].tolist()

# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"

searcher = sm.SimilaritySearcher()
searcher.fit(corpus, cache_path=CACHE_PATH)  # Embeddings cached to disk

# Search (fast - only query embedding computed)
query = "OIKOS TRIPLE ZERO PLAIN HIGH PROTEIN YOGURT\n\nMore of what you want, less of what you don't - 18g of protein 0% fat, 0g added sugars and 0 artificial sweeteners.\n\nAvailable in 32oz large size, Oikos Triple Zero Plain is a deliciously simple way to get the protein you need. Perfect to add to smoothies, parfaits, or enjoy on it's own!\n\n- 18g Protein\n- 0g Added Sugar\n- 0 Artificial Sweeteners\n- 0% Fat\n- Project Non-GMO Verified\n\n32oz Multi-Serve Tub - $5.99\n\nCurrent Oikos Assortment Still Available"
top_results, bottom_results = searcher.search(query, top_n=10, bottom_m=10)

print("Top 5 most similar:")
for text, score in top_results:
    print(f"  {score:.4f}: {text[:80]}...")

print("\nBottom 5 least similar:")
for text, score in bottom_results:
    print(f"  {score:.4f}: {text[:80]}...")

Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 8037 embeddings loaded from cache
Top 5 most similar:
  0.9960: Oikos Triple Zero Plain High Protein Yogurt More of what you want, less of what ...
  0.9960: Oikos Triple Zero Plain High Protein Yogurt More of what you want, less of what ...
  0.9369: Oikos Anything but Plain High Protein Yogurt More of what you want, less of what...
  0.9369: Oikos Anything but Plain High Protein Yogurt More of what you want, less of what...
  0.8461: OIKOS Triple Zero Mocha Flavored Yogurt STRONGER MAKES EVERYTHING BETTER® OIKOS ...
  0.8185: OIKOS PRO+ FUEL HIGH-PROTEIN YOGURT WITH COMPLEX CARBS TO FUEL YOU Try new Oikos...
  0.8185: OIKOS PRO+ FUEL HIGH-PROTEIN YOGURT WITH COMPLEX CARBS TO FUEL YOU Try new Oikos...
  0.8113: OIKOS PRO BI-LAYER HIGH-PROTEIN YOGURT WITH A DELIGHTFUL CREAM TO REWARD YOUR EF...
  0.8113: OIKOS PRO BI-LAYER HIGH-PROTEIN YOGURT WITH A DELIGHTFUL CREAM TO REWARD YOUR 

# people like or agains 

In [19]:
import pandas as pd
import json
import os

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI  # Requires langchain-openai package
from dotenv import load_dotenv
load_dotenv()
lite_llm_key_all = os.getenv('LITE_LLM_KEY_ALL')

llm_model = ChatOpenAI(base_url='https://ipsos.litellm-prod.ai/',model='gpt-5', temperature=0.1, api_key=lite_llm_key_all)

1. giving a concept
2. it's needs, chain of thought, would u think he will like it ?  
3. it shall be able to process a list 
that's it 

In [11]:
from models import need_filter as nf

In [14]:
import importlib
importlib.reload(nf)

<module 'models.need_filter' from 'c:\\Users\\Yuding.Duan\\OneDrive - Ipsos\\3. self_projects\\llm_synthetic\\combination\\..\\models\\need_filter.py'>

In [15]:
import random
# random individual 
random.seed(42)  # For reproducibility
n = 10
ids = list(transformed_respondent.keys()); selected_ids = random.sample(ids, n)
# kpi 
kpis = random.choices(["relevance", "differentiation", "believability"], k=n)

# concepts 
concepts = random.choices(all_us_food_concepts['ConceptText'].tolist(), k=n)

In [ ]:
# Prepare batch items

##### this is only for kinda dataframe generation. 

reasoning = False
items = []
qneeds_texts = []

for resp_id, kpi, concept in zip(selected_ids, kpis, concepts):
    respondent_info = transformed_respondent[resp_id]

    # Get qneeds for display
    qneeds = []
    for q in ['qneed2', 'qneed3']:
        if q in respondent_info:
            qneeds.append(f"{respondent_info[q]['cate']}: {respondent_info[q]['comment']}")
    qneeds_texts.append("\n".join(qneeds))

    items.append({
        "new_concept": concept,
        "kpi_type": kpi,
        "system_info": respondent_info,
        "return_reasoning": reasoning
    })

# Run batch processing concurrently (use await in notebooks)
print(f"Processing {len(items)} items concurrently...")
batch_results = await nf.ai_filter_batch_async(
    items,
    max_concurrency=4,
    show_progress=True,
    progress_desc="AI filter"
)
print("Done!")

# Collect results
results = {
    'respondent_id': selected_ids,
    'respondent_needs': qneeds_texts,
    'kpi_type': kpis,
    'concept': [c[:500] for c in concepts],  # truncate for readability
    'predicted_answer': [r['answer'] if reasoning else r for r in batch_results],
    'reasoning': [r['reasoning'] if reasoning else '' for r in batch_results]
}

# Create DataFrame and export to Excel
results_df = pd.DataFrame(results)
output_path = "../data/sample_ai_filter_results_reasoning.xlsx"
results_df.to_excel(output_path, index=False)
print(f"Results saved to {output_path}")

Processing 10 items concurrently...


AI filter: 100%|██████████| 10/10 [00:18<00:00,  1.88s/it]

Done!
Results saved to ../data/sample_ai_filter_results_reasoning.xlsx


# go for the whole process

**prepare all the data**

- food_concepts  
- all_us_food_concepts  
- transformed_respondent

In [114]:
us_food_cates = list(set(all_us_food_concepts.dropna(subset=['CAT2'])['CAT2']))


# Example usage with caching:
corpus = us_food_cates[:]

# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"

searcher_cate = sm.SimilaritySearcher()
searcher_cate.fit(corpus, cache_path=CACHE_PATH)

# top_results, bottom_results = searcher_cate.search(query, top_n=5, bottom_m=0)

Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 22 embeddings loaded from cache


In [84]:
response_table = pd.read_excel(response_root + "response_table_us_food.xlsx")

In [98]:
kpi_mapping = {
    "relevance": "RelFlag",
    "differentiation": "DiffFlag",
    "believability": "BelFlag"
}

let's go

In [96]:
# minimum unit 
id = "101daf40-ed83-11ee-905c-7d576dd0d5d9"
kpi = "relevance"
mask = (response_table['question'] == kpi) &(response_table['ids']==id)
sub = response_table[mask]
sub

,id,ids,concept,question,answer
0,6,101daf40-ed83-11ee-905c-7d576dd0d5d9,4,relevance,yes
1,6,101daf40-ed83-11ee-905c-7d576dd0d5d9,12,relevance,yes


In [174]:
item = sub.iloc[1]

In [175]:
query = food_concepts[str(item['concept'])]['concept_Cate']
top_results, bottom_results = searcher_cate.search(query, top_n=5, bottom_m=0)

In [176]:
top_results
cate_match_bound = 0.6
suitable_cates = set([k[0] for k in top_results if k[1] >= cate_match_bound])

In [177]:
# tactic determine

# this is to do the opposite one
out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
filted_concepts = all_us_food_concepts[out_mask]

In [178]:
corpus = list(filted_concepts['ConceptText'])

# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"

searcher_concept = sm.SimilaritySearcher()
searcher_concept.fit(corpus, cache_path=CACHE_PATH)

query = food_concepts[str(item['concept'])]['concept_content']
top_results, bottom_results = searcher_concept.search(query, top_n=5, bottom_m=5)

Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 27 embeddings loaded from cache


In [186]:
lower_bound = 0.3

candidate = [item[0] for item  in bottom_results if item[1]<= lower_bound]

In [ ]:
items = []
respondent_info = transformed_respondent[id]
reasoning = False

for concept in candidate:
    items.append({
        "new_concept": concept,
        "kpi_type": kpi,
        "system_info": respondent_info,
        "return_reasoning": reasoning
    })

# Run batch processing concurrently (use await in notebooks)
print(f"Processing {len(items)} items concurrently...")
batch_results = await nf.ai_filter_batch_async(
    items,
    max_concurrency=4,
    show_progress=True,
    progress_desc="AI filter"
)
print("Done!")

Processing 5 items concurrently...


AI filter: 100%|██████████| 5/5 [00:23<00:00,  4.76s/it]

Done!


In [191]:
batch_results

[{'answer': 'yes',
  'reasoning': '1) I prioritize low calories, sugar, and especially sodium; I also look for natural/less processed foods, and I enjoy unique soup flavors and trying new products. \n2) This citrus chicken broth sounds like a unique twist that could brighten soups, which fits my interest in novel flavors and low-calorie bases. However, it doesn’t address my high-protein need, and the concept doesn’t mention low sodium or clean ingredients. If it’s high in sodium or uses artificial “citrus flavor,” it would be a turnoff; if there’s a low-sodium, real-citrus version, it would fit well.\n3) Overall, it’s relevant and appealing for experimentation, but my purchase would depend on sodium level and ingredient quality.'},
 {'answer': 'no',
  'reasoning': '1) My needs: low calories, sugar, and sodium; high protein after gastric sleeve; natural ingredients. I like trying unique flavors, especially unique soups. \n2) The concept is vegan chicken/beef-style broth. That could fit 